## Flags and vars

In [ ]:
# MODIFIED
SaveTrendPlots = True
SaveTrendComparisonsPlots = True
SaveTierMaps = True
SaveSlopeMap = True
SaveHistogram = True

YearsToAverage = 10 # years to average when computing end-start diff

## Install packages

In [ ]:
pip install geopandas

In [ ]:
pip install scikit-learn

## Import standard libraries

In [ ]:
import os
import sys
# append coeqwal packages to path
sys.path.append('./coeqwalpackage')
import datetime as dt
import pandas as pd
import numpy as np
import cqwlutils as cu
import re
import matplotlib.pyplot as plt
import math
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from typing import Optional
from sklearn.linear_model import LinearRegression
import matplotlib.colors as mcolors
from matplotlib.ticker import FixedLocator, FixedFormatter

%matplotlib inline



## Import custom modules

In [ ]:
# Import custom modules
from coeqwalpackage.metrics import read_in_df, add_water_year_column, compute_cv_df
from coeqwalpackage.tier import (
    build_gw_timeseries,
    compute_baseline_percent,
    compute_wba_trends,
    assign_tiers_from_trends,
    parse_wresl_mappings,
    load_gw1_df,
    normalize_id,
    normalize_storage_col,
    normalize_wba_name
)
from coeqwalpackage.cqwlutils import find_repo_root, pad_index
import plotting as pu
from plotting import plot_trendline_comparison_from_matrix

# Specify baseline scenario and other constants

In [ ]:
baseline_scenario = "s0002"
end_year_override = {"s0006", "s0007", "s0008", "s0009", "s0010"} #scenarios that end before others
drop_threshold = 1000
start_year = 1960
start_date = pd.Timestamp("1990-01-01")
end_date = pd.Timestamp("2000-12-31")
slope_start_date="1992-09-30"
window_start = pd.Timestamp("1973-10-31") # GW simulation start and end
window_end   = pd.Timestamp("2015-09-30")
monthly_ft_filename = "GroundWater_Levels_Monthly.csv"
annual_ft_filename = "GroundWater_Levels_Annual.csv"
monthly_af_filename = "GroundWater_Volumes_Monthly.csv"
annual_af_filename = "GroundWater_Volumes_Annual.csv"
monthly_percent_filename = "GroundWater_BaselinePercent_Monthly.csv"
annual_percent_filename = "GroundWater_BaselinePercent_Annual.csv"
monthly_percent_cv_filename = "GroundWater_BaselinePercent_CV_Monthly.csv"
annual_percent_cv_filename = "GroundWater_BaselinePercent_CV_Annual.csv"
monthly_ft_cv_filename = "GroundWater_Levels_CV_monthly.csv"
annual_ft_cv_filename = "GroundWater_Levels_CV_annual.csv"
monthly_af_cv_filename = "GroundWater_Volumes_CV_monthly.csv"
annual_af_cv_filename = "GroundWater_Volumes_CV_annual.csv"
trend_filename = "GroundWater_Trends_ft_per_month.csv"
diff_filename = "GroundWater_AvgEndStartDiff_ft.csv"
GWregionIndex_filename = "CalSim3GWregionIndex.wresl"
WBAIndex_filename = "CalSim3_WBA.csv"
WBAStorage_filename = "20250908draft_C2VSim_73-15_WBA_Storage.csv"
GWwreslmapping_filename = "groundwater_wresl_mapping.csv"
tier_filename = "GroundWater_Tiers.csv"
shapefile_filename = "./shapefiles/i12_CalSim3Model_WaterBudgetAreas_20221021.shp"
all_metrics_filename = "groundwater_all_metrics.csv"

## Create directories

In [ ]:
# find_repo_root is now imported from coeqwalpackage.cqwlutils
repo_root = find_repo_root()

tier_output_dir = repo_root / "CalSim3_Model_Runs" / "Scenarios" / "Performance_Metrics" / "Tiered_Outcome_Measures" / "Groundwater"
tier_output_dir.mkdir(parents=True, exist_ok=True)
data_output_dir = repo_root / "CalSim3_Model_Runs" / "Scenarios" / "Performance_Metrics"/ "Metrics" / "Groundwater"
data_output_dir.mkdir(parents=True, exist_ok=True)
trends_output_dir = tier_output_dir / "TrendPlots"
trends_output_dir.mkdir(parents=True, exist_ok=True)
trends_comparisons_output_dir = tier_output_dir / "TrendComparisonPlots"
trends_comparisons_output_dir.mkdir(parents=True, exist_ok=True)
tiers_output_dir = tier_output_dir / "Tiers"
tiers_output_dir.mkdir(parents=True, exist_ok=True)
tiers_plots_dir = tier_output_dir / "TierPlots"
tiers_plots_dir.mkdir(parents=True, exist_ok=True)

## Set paths

In [ ]:
base_dir = os.path.abspath(".")
wba_storage_csv_path = os.path.join(base_dir, WBAStorage_filename)
output_mapping_csv_path = os.path.join(base_dir, GWwreslmapping_filename)
wresl_path = os.path.join(base_dir, GWregionIndex_filename)
wba_csv_path = os.path.join(base_dir, WBAIndex_filename)
tier_output_path = os.path.join(tiers_output_dir, tier_filename)
shapefile_path = os.path.join(base_dir, shapefile_filename)

## Define contol file name

In [ ]:
CtrlFile = 'CalSim3GroundWaterDataExtractionInitFile_v1.xlsx'
CtrlTab = 'Init'

## Read from control file

In [ ]:
ScenarioListFile, ScenarioListTab, ScenarioListPath, GW1DssNamesOutPath, GW2DssNamesOutPath, ScenarioIndicesOutPath, DssDirsOutPath, VarListPath, VarListFile, VarListTab, VarOutPath, DataOutPath, ConvertDataOutPath, ExtractionSubPath, DemandDeliverySubPath, ModelSubPath, GroupDataDirPath, ScenarioDir, GW1DssMin, GW1DssMax, GW2DssMin, GW2DssMax, NameMin, NameMax, DirMin, DirMax, IndexMin, IndexMax, StartMin, StartMax, EndMin, EndMax, VarMin, VarMax, DemandFilePath, DemandFileName, DemandFileTab, DemMin, DemMax, InflowOutSubPath, InflowFilePath, InflowFileName, InflowFileTab, InflowMin, InflowMax = cu.read_init_file(CtrlFile, CtrlTab)

In [ ]:
print([ScenarioListFile, ScenarioListTab, ScenarioListPath, GW1DssNamesOutPath, GW2DssNamesOutPath, ScenarioIndicesOutPath, DssDirsOutPath, VarListPath, VarListFile, VarListTab, VarOutPath, DataOutPath, ConvertDataOutPath, ExtractionSubPath, DemandDeliverySubPath, ModelSubPath, GroupDataDirPath, ScenarioDir, GW1DssMin, GW1DssMax, GW2DssMin, GW2DssMax, NameMin, NameMax, DirMin, DirMax, IndexMin, IndexMax, StartMin, StartMax, EndMin, EndMax, VarMin, VarMax, DemandFilePath, DemandFileName, DemandFileTab, DemMin, DemMax, InflowOutSubPath, InflowFilePath, InflowFileName, InflowFileTab, InflowMin, InflowMax])


## Check for output directory and create if necessary (not necessary)

In [ ]:
# check if output directory exists
if not os.path.exists(GroupDataDirPath):
    # print warning
    print("Warning: directory " + GroupDataDirPath + " does not exists and will be created")
    
    # Create the directory
    os.makedirs(GroupDataDirPath)


## Define Nan Values

In [ ]:
# NaN values as defined by CalSim3
Nan1 = -901
Nan2 = -902

## Read indeces, dss names, directory names, start and end dates, time range (not necessary)

In [ ]:
gw1dsshdr, gw1dssname = cu.read_from_excel(ScenarioListPath, ScenarioListTab, GW1DssMin, GW1DssMax, hdr=True)
gw1dss_names = []
for i in range(len(gw1dssname)):
    gw1dss_names.append(gw1dssname[i][0])
gw1dss_names

In [ ]:
gw2dsshdr, gw2dssname = cu.read_from_excel(ScenarioListPath, ScenarioListTab, GW2DssMin, GW2DssMax, hdr=True)
gw2dss_names = []
for i in range(len(gw2dssname)):
    gw2dss_names.append(gw2dssname[i][0])
gw2dss_names

In [ ]:
indexhdr, index_name = cu.read_from_excel(ScenarioListPath, ScenarioListTab, IndexMin, IndexMax, hdr=True)
index_names = []
for i in range(len(index_name)):
    if index_name[i][0] is not None and index_name[i][0] != 'None':
        index_names.append(index_name[i][0])
index_names

In [ ]:
studyhdr, study_name = cu.read_from_excel(ScenarioListPath, ScenarioListTab, NameMin, NameMax, hdr=True)
study_names = []
for i in range(len(study_name)):
    study_names.append(study_name[i][0])
study_names

In [ ]:
dirhdr, dir_name = cu.read_from_excel(ScenarioListPath, ScenarioListTab, DirMin, DirMax, hdr=True)
dir_names = []
for i in range(len(dir_name)):
    dir_names.append(dir_name[i][0])
dir_names

In [ ]:
starthdr, start_date = cu.read_from_excel(ScenarioListPath, ScenarioListTab, StartMin, StartMax, hdr=True)
start_dates = []
for i in range(len(start_date)):
    if start_date[i][0] is not None and start_date[i][0] != 'None':
        start_dates.append(start_date[i][0])
datetime_start_dates = pd.to_datetime(start_dates)
# turns out that dss reading library wands a dt datetime, not pd datetime
dt_datetime_start_dates = [dt.to_pydatetime() for dt in datetime_start_dates]

In [ ]:
endhdr, end_date = cu.read_from_excel(ScenarioListPath, ScenarioListTab, EndMin, EndMax, hdr=True)
end_dates = []
for i in range(len(end_date)):
    if end_date[i][0] is not None and end_date[i][0] != 'None':
        end_dates.append(end_date[i][0])
# turns out that dss reading library wands a dt datetime, not pd datetime
datetime_end_dates = pd.to_datetime(end_dates)
dt_datetime_end_dates = [dt.to_pydatetime() for dt in datetime_end_dates]

In [ ]:
min_datetime = min(dt_datetime_start_dates)
print('Min time: ')
print(min_datetime)
max_datetime = max(dt_datetime_end_dates)
print('Max time: ')
print(max_datetime)


## Read variables list (not necessary)

In [ ]:
# get vars
hdr, vars = cu.read_from_excel(VarListPath, VarListTab,VarMin,VarMax,hdr=True)
gw1var_df = pd.DataFrame(data=vars, columns=hdr)
gw1var_df

In [ ]:
print(gw1var_df.head(20))   # show first 20 rows
print(gw1var_df.columns)    # see what metadata is included
print(gw1var_df.shape)      # rows × cols


## Read the compund data from CSV to df

In [ ]:
# read the dataframe from CSV
print('Reading ' + DataOutPath)
gw1_df, gw1dss_names = read_in_df(DataOutPath,GW1DssNamesOutPath)

In [ ]:
print("gw1dss_names:")
gw1dss_names

In [ ]:
print("gw1_df:")
gw1_df

## Drop the LT:E999 columns

In [ ]:
mask = ~gw1_df.columns.to_frame().apply(lambda col: col.astype(str).str.contains('LT:E999')).any(axis=1)
gw1_df = gw1_df.loc[:, mask.values]

In [ ]:
print("new gw1_df:")
gw1_df

## Add water year column to df

In [ ]:
# add_water_year_column() is imported from coeqwalpackage.metrics

In [ ]:
gw1_df = add_water_year_column(gw1_df)

In [ ]:
print("gw1_df with water year column:")
gw1_df

## End of initialization

In [ ]:
print('Done Initializing!')

## Map SR to WBA

In [ ]:
# MODIFIED
# Read the area data (CSV)
wba_df = pd.read_csv(wba_csv_path)

# Check what columns are present
print("Columns in WBA Area CSV:", wba_df.columns)

# Preview key columns
print(wba_df[['fid', 'GIS_Acres']].head())


with open(wresl_path, 'r') as f:
    wresl_lines = f.readlines()

# Extract SRxx → WBAxx or DETAW
sr_to_wba_map = {}
for line in wresl_lines:
    match = re.match(r'\s*indxWBA_(\d+)\s*=\s*(SR\d+)', line)
    if match:
        wba_num, sr_num = match.groups()
        sr_to_wba_map[sr_num] = f'WBA{wba_num}'
    else:
        match_detaw = re.match(r'\s*indxDETAW\s*=\s*(SR\d+)', line)
        if match_detaw:
            sr_num = match_detaw.group(1)
            sr_to_wba_map[sr_num] = 'DETAW'


# # Preview result
# print("\n=== SR to WBA Mapping Preview ===")
# print(mapping_df.head())

# Save mapping to CSV if needed


In [ ]:
with open(wresl_path, 'r') as f:
    lines = f.readlines()

# Parse mappings
sr_to_wba_map = {}

for line in lines:
    line = line.strip()

    # Handle standard: define indxWBA_2 {value 1 }
    match_wba = re.match(r'define\s+indxWBA_([0-9A-Za-z]+)\s+\{value\s+(\d+)\s+\}', line)
    if match_wba:
        wba_id, sr_num = match_wba.groups()
        sr_key = f"SR{int(sr_num):02d}"       # e.g. 1 → SR01
        wba_value = f"WBA{wba_id}"            # e.g. 2 → WBA2
        sr_to_wba_map[sr_key] = wba_value
        continue

    # Handle special case: define indxDETAW {value 42 }
    match_detaw = re.match(r'define\s+indxDETAW\s+\{value\s+(\d+)\s+\}', line)
    if match_detaw:
        sr_num = match_detaw.group(1)
        sr_key = f"SR{int(sr_num):02d}"       # e.g. 42 → SR42
        sr_to_wba_map[sr_key] = "DETAW"

# Convert to DataFrame
mapping_df = pd.DataFrame(list(sr_to_wba_map.items()), columns=["SR_number", "WBA_name"])
print(mapping_df.head(10))  # check the DETAW row

mapping_df.to_csv("sr_to_wba_mapping.csv", index=False)


In [ ]:

with open(wresl_path, "r") as file:
    lines = file.readlines()

mapping_records = []

i = 1
for line in lines:
    line = line.strip()
    # print("line " + str(i) + ":")
    # print(line)
    match_wba = re.match(r'define\s+indxWBA_([0-9A-Za-z]+)\s+\{value\s+(\d+)\s+\}', line)
    # print("match_wba " + str(i) + ":")
    # print(match_wba)
    
    if match_wba:
        wba_id, sr_num = match_wba.groups()
        # print("wba_id " + str(i) + ":")
        # print(wba_id)
        # print("sr_num " + str(i) + ":")
        # print(sr_num)
        mapping_records.append({
            "Subregion_ID": f"SR{int(sr_num):02d}",
            "WBA_ID": f"WBA{wba_id}"
        })
        continue
    i = i + 1
    match_detaw = re.match(r'define\s+indxDETAW\s+\{value\s+(\d+)\s+\}', line)
    if match_detaw:
        sr_num = match_detaw.group(1)
        mapping_records.append({
            "Subregion_ID": f"SR{int(sr_num):02d}",
            "WBA_ID": "DETAW"
        })

# print("mapping_records:")
# print(mapping_records)

wresl_df = pd.DataFrame(mapping_records).sort_values("Subregion_ID")
display(wresl_df)

wresl_df.to_csv(output_mapping_csv_path, index=False)
print(f"Saved WRESL mapping to: {output_mapping_csv_path}")


## Monthly, Annual Data and Trend 

In [ ]:
# Build GW metrics using modular tier.py functions
# Step 1: Build FT and AF timeseries
ft_monthly, ft_annual, af_monthly, af_annual = build_gw_timeseries(
    gw_csv_path=DataOutPath,
    wba_csv_path=wba_csv_path,
    wba_storage_csv_path=wba_storage_csv_path,
    mapping_df=wresl_df,  # Created in cell 51
    window_start=window_start,
    window_end=window_end,
    start_year=start_year
)

# Step 2: Compute baseline % for AF
af_pct_monthly = compute_baseline_percent(af_monthly, baseline_scenario=baseline_scenario)
af_pct_annual = compute_baseline_percent(af_annual, baseline_scenario=baseline_scenario)

# Step 3: Compute CV for each timeseries
ft_cv_monthly = compute_cv_df(ft_monthly)
ft_cv_annual = compute_cv_df(ft_annual)
af_cv_monthly = compute_cv_df(af_monthly)
af_cv_annual = compute_cv_df(af_annual)
af_pct_cv_monthly = compute_cv_df(af_pct_monthly)
af_pct_cv_annual = compute_cv_df(af_pct_annual)

# Save all outputs
ft_monthly.to_csv(os.path.join(data_output_dir, monthly_ft_filename))
ft_annual.to_csv(os.path.join(data_output_dir, annual_ft_filename))
af_monthly.to_csv(os.path.join(data_output_dir, monthly_af_filename))
af_annual.to_csv(os.path.join(data_output_dir, annual_af_filename))
af_pct_monthly.to_csv(os.path.join(data_output_dir, monthly_percent_filename))
af_pct_annual.to_csv(os.path.join(data_output_dir, annual_percent_filename))
ft_cv_monthly.to_csv(os.path.join(data_output_dir, monthly_ft_cv_filename))
ft_cv_annual.to_csv(os.path.join(data_output_dir, annual_ft_cv_filename))
af_cv_monthly.to_csv(os.path.join(data_output_dir, monthly_af_cv_filename))
af_cv_annual.to_csv(os.path.join(data_output_dir, annual_af_cv_filename))
af_pct_cv_monthly.to_csv(os.path.join(data_output_dir, monthly_percent_cv_filename))
af_pct_cv_annual.to_csv(os.path.join(data_output_dir, annual_percent_cv_filename))

# combined_monthly/annual are used downstream (trends, plots, etc.)
combined_monthly = ft_monthly
combined_annual = ft_annual

print(f"FT Monthly shape: {ft_monthly.shape}")
print(f"FT Annual shape: {ft_annual.shape}")
print(f"AF Monthly shape: {af_monthly.shape}")
print(f"AF Annual shape: {af_annual.shape}")
print(f"AF % Monthly shape: {af_pct_monthly.shape}")
print(f"AF % Annual shape: {af_pct_annual.shape}")
print(f"\nAll GW metrics saved to: {data_output_dir}")


In [ ]:
# Compute trends (slopes) and diffs (end-start averages)
trend_matrix, diff_matrix = compute_wba_trends(
    combined_monthly=combined_monthly,
    trends_output_dir=data_output_dir,
    trend_filename=trend_filename,
    diff_filename=diff_filename,
    years_to_average=YearsToAverage
)

trends_out_path = os.path.join(data_output_dir, trend_filename)

print(f"Trend matrix shape: {trend_matrix.shape}")
print(f"Diff matrix shape: {diff_matrix.shape}")
trend_matrix.head()

# Plot histograms to find a natural break in the trends of baseline scenario

## Specify number of bins

In [ ]:
nBins = 50

## Plot baseline trend histogram

In [ ]:
# MODIFIED
if SaveHistogram:
    baseline_data = trend_matrix.loc[baseline_scenario].dropna()
    
    # --- Plot histogram ---
    bins = 20  # you can set nBins here
    plt.figure(figsize=(10, 6))
    counts, bin_edges, _ = plt.hist(baseline_data, bins=bins, edgecolor='black')
    
    save_name = f"TrendsHistogram_{baseline_scenario}.png"
    save_path = os.path.join(trends_output_dir, save_name)
    
    # Add more x-axis ticks using bin edges
    plt.xticks(np.round(bin_edges, 4), rotation=45)
    plt.title(f"Histogram of Monthly Trends (ft/month) for {baseline_scenario}")
    plt.xlabel("Slope Value (ft/month)")
    plt.ylabel("Frequency")
    plt.grid(True)
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"✓ Saved: {save_path}")

## Specify clipping quantiles

In [ ]:
lQuant = 0
hQuant = 0.5

## Plot clipped baseline trend histogram

In [ ]:
if SaveHistogram:

    baseline_data = trend_matrix.loc[baseline_scenario].dropna()
    
    lVal, hVal = np.quantile(baseline_data.values, [lQuant, hQuant])
    
    # --- Clip data ---
    clipped_data = baseline_data.values.clip(lVal, hVal)
    
    # --- Plot histogram ---
    bins = 20  # or nBins if you’ve defined it elsewhere
    plt.figure(figsize=(10, 6))
    counts, bin_edges, _ = plt.hist(clipped_data, bins=bins, edgecolor='black')
    
    save_name = f"ClippedTrendsHistogram_{baseline_scenario}.png"
    save_path = os.path.join(trends_output_dir, save_name)
    
    # Add more x-axis ticks using bin edges
    plt.xticks(np.round(bin_edges, 4), rotation=45)
    plt.title(f"Histogram of Monthly Trends (ft/month) for {baseline_scenario} (after clipping)")
    plt.xlabel("Slope Value (ft/month)")
    plt.ylabel("Frequency")
    plt.grid(True)
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"✓ Saved: {save_path}")

## Notes: Where to put the break? What threshold to use to distinguish moderate from severe? Propose: -0.015; For the GW model (s0000) we tried -0.0025; For s0002, trying -0.0075

In [ ]:
severe_decline_threshold=-0.0075

## Plot trends

In [ ]:
# MODIFIED
if SaveTrendPlots:
    scenarios_to_plot = sorted({baseline_scenario, *index_names})
    print("Scenarios to plot:")
    print(scenarios_to_plot)
    drop_threshold = 1000
    start_year = 1960
    
    # combined_monthly should already be loaded in your workspace
    gw1_df_filtered = combined_monthly.copy()
    
    for col in gw1_df_filtered.columns:
        if "_s" not in col:
            continue
    
        wba_id, scenario = col.split("_")
        if scenario not in scenarios_to_plot:
            continue
    
        ts = gw1_df_filtered[col].dropna()
        ts = ts[ts.index >= pd.Timestamp(f"{start_year}-01-01")]
    
        diffs = ts.diff()
        drop_indices = diffs[diffs < -drop_threshold].index
        if not drop_indices.empty:
            cutoff_idx = drop_indices[0]
            ts = ts[ts.index <= cutoff_idx]
            drop_year = cutoff_idx.year
        else:
            drop_year = 2015  # clipped already
    
        if len(ts) < 2:
            continue  # skip if not enough data
    
        # Fit trendline
        x = (ts.index - ts.index[0]).days / 365.25
        y = ts.values
        slope, intercept = np.polyfit(x, y, 1)
        trend = slope * x + intercept
    
        plt.figure(figsize=(10, 4))
        plt.plot(ts.index, y, label="Observed", marker="o")
        plt.plot(ts.index, trend, linestyle="--", label=f"Trend (slope={slope:.6f})")
        
        # Ensure title not clipped by adding pad
        plt.title(f"{wba_id} under {scenario} (end year: {drop_year})", pad=20)
        plt.xlabel("Date")
        plt.ylabel("Groundwater Storage (FT)")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        
        # Save
        save_name = f"{scenario}_{wba_id}.png"
        save_path = os.path.join(trends_output_dir, save_name)
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.show()
        plt.close()
        print(f"✓ Saved: {save_path}")
    

## Trend Comparison

In [ ]:
# plot_trendline_comparison_from_matrix is now imported from plotting module

time_index = pd.date_range(start="1973-10-31", end="2015-09-30", freq="ME")
t_months = np.arange(len(time_index))  # 0,1,2,... months

if SaveTrendComparisonsPlots:
    all_scenarios = [sc for sc in trend_matrix.index if sc != baseline_scenario]
    for sc in all_scenarios:
        plot_trendline_comparison_from_matrix(
            trend_matrix=trend_matrix,
            scenario_code=sc,
            baseline_code=baseline_scenario,
            time_index=time_index,
            t_months=t_months,
            save_dir=trends_comparisons_output_dir
        )

## Compute tiers

In [ ]:
# assign_tiers_from_trends() is imported from coeqwalpackage.tier

# Call the function
tier_matrix = assign_tiers_from_trends(
    trend_matrix,
    baseline=baseline_scenario,
    output_dir=tiers_output_dir,
    filename=tier_filename,
    severe_decline_threshold=severe_decline_threshold
)

## Tier maps

In [ ]:
# MODIFIED
if SaveTierMaps:
    wba_shp = gpd.read_file(shapefile_path)
    wba_shp["WBA_ID"] = wba_shp["WBA_ID"].str.strip()
    
    tier_df = pd.read_csv(tier_output_path, index_col=0)
    
    new_columns = {}
    for col in tier_df.columns:
        if col.startswith("WBA"):
            suffix = col[3:]
            if suffix.isdigit():
                new_columns[col] = suffix.zfill(2)
            else:
                digits = ''.join(filter(str.isdigit, suffix)).zfill(2)
                letter = ''.join(filter(str.isalpha, suffix))
                new_columns[col] = digits + letter
    tier_df.rename(columns=new_columns, inplace=True)
    
    tier_colors = {
        1: "#B5CDA3",  # olive green
        2: "#8FBBD9",  # soft dusty blue
        3: "#E6C27A",  # warm khaki
        4: "#D97B6D",  # soft brick red (worst)
    }
    
    label_shifts = {
        "17STOT": 0.015,
        "71TOT": 0.015,
        "22TOT": 0.015,
        "50TOT": -0.015,
        "21TOT": -0.015,
        "12TOT": -0.015,
    }
    
    for scenario in tier_df.index:
        if scenario == baseline_scenario:  # skip baseline
            continue
    
        # Map tier data to shapefile
        tier_series = tier_df.loc[scenario]
        tier_map = wba_shp.copy()
        tier_map["GroundwaterTier"] = tier_map["WBA_ID"].map(tier_series.to_dict())
    
        # Plot base
        fig, ax = plt.subplots(figsize=(8, 10))
        for tier_val, color in tier_colors.items():
            subset = tier_map[tier_map["GroundwaterTier"] == tier_val]
            if not subset.empty:
                subset.plot(
                    ax=ax,
                    color=color,
                    edgecolor='black',
                    linewidth=0.3,
                    label=f"Tier {tier_val}"
                )
    
        for idx, row in tier_map.iterrows():
            if pd.notna(row["GroundwaterTier"]):
                x, y = row.geometry.centroid.x, row.geometry.centroid.y
                wba_id = row["WBA_ID"]
                shift = label_shifts.get(wba_id, 0)
                ax.text(
                    x, y + shift, wba_id,
                    fontsize=7, weight='bold', ha='center'
                )
    
        ax.set_title(f"Groundwater Tiers for Scenario {scenario}", fontweight="bold")
        ax.axis("off")
    
        # Add legend
        legend_handles = [mpatches.Patch(color=color, label=f"Tier {tier}")
                          for tier, color in tier_colors.items()]
        ax.legend(handles=legend_handles, title="Tier", loc="lower left", frameon=True)
    
        # Save and show
        save_path = os.path.join(tiers_plots_dir, f"GroundWaterTiers_{scenario}.png")
        plt.tight_layout()
        plt.savefig(save_path, dpi=300)
        plt.show()
    
        print(f"✓ Saved map for {scenario} to: {tiers_plots_dir}")




## Slope rank map

In [ ]:
# pad_index is now imported from coeqwalpackage.cqwlutils

if SaveSlopeMap:
    wba_shp = gpd.read_file(shapefile_path)
    wba_shp["WBA_ID"] = wba_shp["WBA_ID"].str.strip().str.upper()
    
    trend_csv = trends_out_path
    
    df = pd.read_csv(trend_csv, index_col=0)
    
    scenarios_to_plot = [baseline_scenario]
    
    for scenario in scenarios_to_plot:
        if scenario not in df.index:
            print(f"⚠ Scenario {scenario} not found in {trend_csv}")
            continue
    
        baseline_data = df.loc[scenario].dropna()
    
        # --- Clean WBA names ---
        baseline_data.index = (
            baseline_data.index
            .str.replace("WBA", "", regex=False)
            .str.replace(":TOT", "", regex=False)
            .str.replace("TOT", "", regex=False)
            .str.lstrip("0")
            .str.strip()
            .str.upper()
        )
        baseline_data.index = baseline_data.index.map(pad_index)
    
        slope_rank = baseline_data.rank().astype(int)
        slope_map = wba_shp.copy()
        slope_map["Slope"] = slope_map["WBA_ID"].map(baseline_data.to_dict())
        slope_map["SlopeRank"] = slope_map["WBA_ID"].map(slope_rank.to_dict())
    
        num_ranks = slope_rank.nunique()
        cmap = plt.colormaps.get_cmap("coolwarm_r").resampled(num_ranks)
        norm = mcolors.Normalize(vmin=1, vmax=num_ranks)
    
        fig, ax = plt.subplots(figsize=(8, 10))
        slope_map.plot(
            column="SlopeRank",
            cmap=cmap,
            linewidth=0.3,
            edgecolor='black',
            ax=ax,
            legend=False,
            norm=norm
        )
    
        # Add labels
        for idx, row in slope_map.iterrows():
            if pd.notna(row["SlopeRank"]):
                x, y = row.geometry.centroid.x, row.geometry.centroid.y
                ax.text(x, y, row["WBA_ID"], fontsize=7, weight='bold', ha='center')
    
        # Add colorbar with slope values
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm._A = []
        rank_to_slope = slope_map.dropna(subset=["Slope", "SlopeRank"]) \
                                 .groupby("SlopeRank")["Slope"].mean().sort_index()
        tick_locs = list(rank_to_slope.index)
        tick_labels = [f"{s:.6f}" for s in rank_to_slope.values]
    
        cbar = fig.colorbar(sm, ax=ax, orientation="vertical", ticks=tick_locs)
        cbar.set_label("Slope Value (ft/month)")
        cbar.ax.yaxis.set_major_locator(FixedLocator(tick_locs))
        cbar.ax.yaxis.set_major_formatter(FixedFormatter(tick_labels))
    
        ax.set_title(f"Slope Rank Map for Scenario {scenario}", fontweight="bold", pad=15)
        ax.axis("off")
        plt.tight_layout()
    
        save_path = os.path.join(tiers_plots_dir, f"SlopeRankMap_{scenario}.png")
        plt.savefig(save_path, dpi=300)
        plt.show()
    
        print(f"✓ Saved slope rank map for {scenario} to {tiers_plots_dir}")

## Merge All Metrics

In [ ]:
# Combine key metrics into single DataFrame
# Rename columns to avoid conflicts
tier_renamed = tier_matrix.add_suffix('_Tier')
trend_renamed = trend_matrix.add_suffix('_Slope')
diff_renamed = diff_matrix.add_suffix('_Diff')
cv_ft_renamed = ft_cv_annual.add_suffix('_CV_FT')
cv_af_renamed = af_cv_annual.add_suffix('_CV_AF')

df_combined = pd.concat([
    tier_renamed,
    trend_renamed,
    diff_renamed,
    cv_ft_renamed,
    cv_af_renamed
], axis=1)

# Save combined output
combined_output_path = os.path.join(data_output_dir, all_metrics_filename)
df_combined.to_csv(combined_output_path)
print(f"Combined metrics saved to: {combined_output_path}")
display(df_combined.head())

In [ ]:
print("Done!")